Necessary installs

In [ ]:

!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -U transformers peft accelerate trl datasets huggingface_hub
!pip install -U -q bitsandbytes>=0.46.1

Clear training

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
import json
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


def HGlogin(HGtoken):
  login(token=HGtoken)
  return()

def Lamma_setup():
  # Load Llama 3.1 8B
  model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

  bnb_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.bfloat16,
      bnb_4bit_use_double_quant=True,
  )

  model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
  tokenizer = AutoTokenizer.from_pretrained(model_id)
  tokenizer.pad_token = tokenizer.eos_token
  return(model, tokenizer)




# Define the formatting logic as a simple function
def format_data_for_trainer(example):
    # Replicate the collaborator's message structure
    # 2. Prompt to set up the system role
    prompt = "You are a logic expert specializing in syllogisms. For the following statements find two premises from which the conclusion follows, if there are such premises. You should respond with a single word, without change in capitalization or punctuation, either TRUE, if there are presented premises from which the conclusion follows, or FALSE otherwise. If the premises are presented you should after the word TRUE indicate their position among the santances in the form TRUE, rank of the first premise, rank of the second premise. The sentances are ordered from 0. Here start your syllogism:"

    messages = [
        {"role": "system", "content": prompt.strip()},
        {"role": "user", "content": example['syllogism']},
        {"role": "assistant", "content": f"TRUE, {example["relevant_premises"][0]}, {example["relevant_premises"][1]}" if str(example["validity"]).upper() == "TRUE" else "FALSE"}
    ]

    # Use the tokenizer directly (it will be in your local scope)
    example["text"] = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return example


def LoRa_configuration(model, formatted_train_path, tokenizer):
  # Load your local file from the sidebar
  #dataset = load_dataset("json", data_files="my_syllogisms.jsonl", split="train")
  dataset = load_dataset("json", data_files=formatted_train_path, split="train")

  # 1. LoRA Configuration
  peft_config = LoraConfig(
      r=16,
      lora_alpha=32,
      target_modules="all-linear",
      bias="none",
      task_type="CAUSAL_LM"
  )

  training_args = SFTConfig(
      output_dir="./syllogism_model_checkpoints",
      max_length=512,
      dataset_text_field="text",
      packing=False,
      per_device_train_batch_size=5,
      gradient_accumulation_steps=4,
      learning_rate=2e-4,
      num_train_epochs=3,
      logging_steps=1,
      fp16=not torch.cuda.is_bf16_supported(),
      bf16=torch.cuda.is_bf16_supported(),
  )



  formatted_dataset = dataset.map(format_data_for_trainer)
  print(formatted_dataset)
  # 3. Initialize the Trainer
  trainer = SFTTrainer(
      model=model,
      train_dataset=formatted_dataset,
      peft_config=peft_config,
      args=training_args,
  )
  return trainer



# Verify we are on the GPU
print(f"Working on GPU: {torch.cuda.get_device_name(0)}")




In [ ]:
# Verify we are on the GPU
print(f"Working on GPU: {torch.cuda.get_device_name(0)}")
HGtoken = input('Your HG writing token:')

HGlogin(HGtoken)
model,tokenizer = Lamma_setup()


In [ ]:
#loads the training data and formats them
path_to_training_data = "/content/train_data.json"#input('add path to the training data or add the default one: /content/my_syllogisms.jsonl')
train_dataset=load_dataset("json", data_files=path_to_training_data, split="train")
#training_data_setup(path_to_training_data, "/content/train_data.json")

#loads the test data and formats them
path_to_test_data = "/content/test_data.json"#input('add path to the training data or add the default one: /content/test_data_subtask_1.jsonl')
test_dataset = load_dataset("json", data_files=path_to_test_data, split="train")
#training_data_setup("test_data.json", "output_test_data.jsonl")

#setup the trainer
trainer = LoRa_configuration(model, "/content/train_data.json", tokenizer)


In [ ]:
#train the model
trainer.train()

In [ ]:
#saves the trained weights
trainer.model.save_pretrained("./content/LoRa_weights")
tokenizer.save_pretrained("./content/LoRa_weights")

#stores the trained model
trained_model = trainer.model

Eval cose

In [ ]:
#code for aplying LoRa, from LoRa_files use when you want to apply pretrained LoRa without training.

from peft import PeftModel
from transformers import AutoTokenizer

# 1. Attach the LoRA weights to the base model
# 'model' is your base model already loaded in memory
trained_model = PeftModel.from_pretrained(model, "LoRa_files")

# 2. Load the corresponding tokenizer from the same folder
tokenizer = AutoTokenizer.from_pretrained("LoRa_files")

# 3. (Optional) Merge the weights if you want a single unified model
# only do this if you are finished with training and want faster inference
# model = model.merge_and_unload()

print("✅ LoRA weights successfully attached to the base model.")

In [ ]:
#evaluation code

trained_model.eval()


def format_data_for_tester(example):
    # Replicate the collaborator's message structure
    # 2. Prompt to set up the system role
    prompt = "You are a logic expert specializing in syllogisms. For the following statements find two premises from which the conclusion follows, if there are such premises. You should respond with a single word, without change in capitalization or punctuation, either TRUE, if there are presented premises from which the conclusion follows, or FALSE otherwise. If the premises are presented you should after the word TRUE indicate their position among the santances in the form TRUE, rank of the first premise, rank of the second premise. The sentances are ordered from 0. Here start your syllogism:"

    messages = [
        {"role": "system", "content": prompt.strip()},
        {"role": "user", "content": example['syllogism']},
    ]

    # Use the tokenizer directly (it will be in your local scope)
    example["text"] = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        return_tensors="pt"
    )
    return example

# load the formatted test set you already created
test_dataset = load_dataset("json", data_files="test_data.json", split="train")
formatted_test_dataset=test_dataset.map(format_data_for_tester)

def generate_label(model, tokenizer, syllogism_text, max_new_tokens=8):
    #This "dictionary comprehension" loops through every piece of your input and shoves it onto the same device as the model (usually cuda:0). If you don't do this, Python will crash with a RuntimeError: Expected all tensors to be on the same device.
    inputs = tokenizer(syllogism_text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

    if prediction.startswith("TRUE"):
        return prediction[:10]
    elif prediction.startswith("FALSE"):
        return "FALSE"
    else:
        return prediction

base_correct = 0
tuned_correct = 0
base_predictions = []
tuned_predictions = []
for i, example in enumerate(formatted_test_dataset):
    formated_syllogism = example["text"]
    gold = f"TRUE, {example["relevant_premises"][0]}, {example["relevant_premises"][1]}" if example["validity"] else 'FALSE'

    # base model = same model, adapter temporarily disabled
    with trained_model.disable_adapter():
        base_pred = generate_label(trained_model, tokenizer, formated_syllogism).upper()

    # tuned model = adapter active
    tuned_pred = generate_label(trained_model, tokenizer, formated_syllogism).upper()

    base_predictions.append(base_pred)
    tuned_predictions.append(tuned_pred)

    if base_pred == gold:
        base_correct += 1
    if tuned_pred == gold:
        tuned_correct += 1

    print(
        f"Example {i}: gold={gold} | base={base_pred} | tuned={tuned_pred} | base_cor={base_correct} | tune_cor={tuned_correct}"
    )

n = len(formatted_test_dataset)
base_accuracy = base_correct / n
tuned_accuracy = tuned_correct / n

print("\n--- FINAL RESULTS ---")
print(f"Number of test examples: {n}")
print(f"Base model accuracy:  {base_accuracy:.4f}")
print(f"Tuned model accuracy: {tuned_accuracy:.4f}")
print(f"Improvement:          {tuned_accuracy - base_accuracy:+.4f}")